# Phase 3 — Affinity Predictor

Trains an ensemble of GINEConv models to predict drug-target binding affinity (pKi).

### Lessons from the previous (failed) approach

| Problem | Why it hurt | Our fix |
|---|---|---|
| GCNConv | Ignores bond features, least expressive GNN | GINEConv (uses edge_attr, provably more powerful) |
| MSE + ranking loss | Ranking doesn't care about values → destroys RMSE | Huber loss (robust MSE, directly optimizes RMSE) |
| Group DRO max-loss | Bounces between groups → val RMSE oscillates wildly | WeightedRandomSampler handles balance (already in Phase 1) |
| Checkpoint on SR-CI | Kaggle scores RMSE, not CI | Checkpoint on best val RMSE |
| No LR schedule | Model can't fine-tune late in training | CosineAnnealingWarmRestarts |
| Ignored fingerprints | Phase 2 computed 2048-bit Morgan FPs, never used | fp_proj branch feeds FP into prediction head |
| 3-model ensemble | Limited diversity | 5 models, different seeds |

## 1. Setup

In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINEConv, global_mean_pool
from torch_geometric.loader import DataLoader
from torch.utils.data import WeightedRandomSampler
from lifelines.utils import concordance_index
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


In [2]:
# Shared config across Phase 3/4/5
INPUT_DIR    = 'data/'
MODEL_DIR    = 'saved_models_v2/'
ENSEMBLE_SIZE = 5
HIDDEN_DIM   = 256
ESM_DIM      = 1280
NODE_FEAT    = 29
EDGE_FEAT    = 7
N_GIN_LAYERS = 4
DROPOUT      = 0.2
BATCH_SIZE   = 128

EPOCHS    = 100
PATIENCE  = 20
LR        = 1e-3

os.makedirs(MODEL_DIR, exist_ok=True)

## 2. Model Architecture

In [11]:
class DTA_Model(nn.Module):
    """
    Drug-Target Affinity predictor.
    
    Improvements over the previous GCN approach:
    - GINEConv: uses bond features (edge_attr), more expressive than GCN
    - BatchNorm: stabilizes training, prevents the wild RMSE oscillations
    - Morgan fingerprint branch: uses the 2048-bit FPs computed in Phase 2
    - Dropout throughout: enables MC Dropout for uncertainty in Phase 4
    """

    def __init__(self, node_feat=29, edge_feat=7, esm_dim=1280,
                 hidden_dim=256, n_layers=4, dropout=0.2):
        super().__init__()
        self.dropout = dropout

        # --- Drug graph encoder (GINEConv) ---
        self.edge_proj = nn.Linear(edge_feat, hidden_dim)

        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        for i in range(n_layers):
            in_dim = node_feat if i == 0 else hidden_dim
            mlp = nn.Sequential(
                nn.Linear(in_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
            )
            self.convs.append(GINEConv(mlp, edge_dim=hidden_dim))
            self.bns.append(nn.BatchNorm1d(hidden_dim))

        # --- Drug fingerprint encoder ---
        self.fp_proj = nn.Sequential(
            nn.Linear(2048, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # --- Protein encoder ---
        self.prot_proj = nn.Sequential(
            nn.Linear(esm_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
        )

        # --- Prediction head ---
        # graph(256) + fp(256) + protein(256) = 768
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def get_compound_embedding(self, data):
        """Extract the compound graph embedding (used by Phase 4 for OOD detection)."""
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        edge_emb = self.edge_proj(edge_attr)
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index, edge_emb)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return global_mean_pool(x, batch)

    def forward(self, data):
        # 1. Drug graph
        drug_graph = self.get_compound_embedding(data)  # (B, hidden)
        # 2. Drug fingerprint
        drug_fp = self.fp_proj(data.fp.squeeze(1))      # (B, hidden)
        # 3. Protein
        prot = self.prot_proj(data.target_emb.squeeze(1))  # (B, hidden)
        # 4. Predict
        combined = torch.cat([drug_graph, drug_fp, prot], dim=-1)
        return self.head(combined).squeeze(-1)  # (B,)

## 3. Evaluation

In [12]:
def evaluate(model, loader, device):
    """Returns RMSE, SR-CI, and per-split CIs on the validation set."""
    model.eval()
    all_preds, all_targets = [], []
    all_A, all_B, all_C = [], [], []

    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            preds = model(data)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(data.y.cpu().numpy())
            all_A.extend(data.in_A.cpu().numpy())
            all_B.extend(data.in_B.cpu().numpy())
            all_C.extend(data.in_C.cpu().numpy())

    p, t = np.array(all_preds), np.array(all_targets)
    rmse = np.sqrt(np.mean((p - t) ** 2))

    def safe_ci(targets, preds, mask):
        mask = np.array(mask) == 1
        if mask.sum() < 2: return 0.5
        return concordance_index(targets[mask], preds[mask])

    ci_a = safe_ci(t, p, all_A)
    ci_b = safe_ci(t, p, all_B)
    ci_c = safe_ci(t, p, all_C)
    sr_ci = 0.30 * ci_a + 0.35 * ci_b + 0.35 * ci_c

    return rmse, sr_ci, ci_a, ci_b, ci_c

## 4. Load Data & Train Ensemble

**Why Huber loss?** The Kaggle metric is RMSE, so we want something close to MSE. But real assay data has noisy outliers — a few measurements with huge error can dominate MSE and push the model to overfit noise. Huber loss acts like MSE for small errors (good RMSE) but like MAE for large errors (ignores outliers). Best of both worlds.

In [13]:
print('Loading data...')
train_data = torch.load(os.path.join(INPUT_DIR, 'train_data.pt'))
val_data   = torch.load(os.path.join(INPUT_DIR, 'val_data.pt'))

# Phase 1's weighted sampler balances groups — no need for Group DRO max-loss
train_weights = [d.sample_weight.item() for d in train_data]
sampler = WeightedRandomSampler(train_weights, len(train_weights), replacement=True)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, sampler=sampler, drop_last=True)
val_loader   = DataLoader(val_data,   batch_size=BATCH_SIZE, shuffle=False)

print(f'Train: {len(train_data)} | Val: {len(val_data)} | Batches/epoch: {len(train_loader)}')

Loading data...
Train: 23194 | Val: 2578 | Batches/epoch: 181


In [14]:
criterion = nn.HuberLoss(delta=1.0)

print(f'Training {ENSEMBLE_SIZE}-model ensemble...\n')
all_best_rmse = []

for ens_idx in range(1, ENSEMBLE_SIZE + 1):
    torch.manual_seed(42 + ens_idx * 1000)
    np.random.seed(42 + ens_idx * 1000)

    print(f'=== Model {ens_idx}/{ENSEMBLE_SIZE} ===')

    model = DTA_Model(
        node_feat=NODE_FEAT, edge_feat=EDGE_FEAT, esm_dim=ESM_DIM,
        hidden_dim=HIDDEN_DIM, n_layers=N_GIN_LAYERS, dropout=DROPOUT
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=20, T_mult=2, eta_min=1e-6
    )

    best_rmse = float('inf')
    best_sr_ci = 0.0
    patience_ctr = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss, n_samples = 0, 0

        for batch in tqdm(train_loader, desc=f'Ep {epoch:02d}', leave=False):
            batch = batch.to(device)
            optimizer.zero_grad()
            preds = model(batch)
            loss = criterion(preds, batch.y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            total_loss += loss.item() * batch.num_graphs
            n_samples += batch.num_graphs

        scheduler.step()
        avg_loss = total_loss / n_samples

        rmse, sr_ci, ci_a, ci_b, ci_c = evaluate(model, val_loader, device)

        if epoch % 5 == 0 or epoch == 1:
            print(f'Ep {epoch:03d} | Loss: {avg_loss:.4f} | RMSE: {rmse:.4f} | '
                  f'SR-CI: {sr_ci:.4f} (A:{ci_a:.3f} B:{ci_b:.3f} C:{ci_c:.3f})')

        # Save on best RMSE — this is what Kaggle scores
        if rmse < best_rmse:
            best_rmse = rmse
            best_sr_ci = sr_ci
            patience_ctr = 0
            torch.save(model.state_dict(), os.path.join(MODEL_DIR, f'model_{ens_idx}.pt'))
        else:
            patience_ctr += 1

        if patience_ctr >= PATIENCE:
            print(f'Early stop ep {epoch}. Best RMSE: {best_rmse:.4f}, SR-CI: {best_sr_ci:.4f}')
            break

    all_best_rmse.append(best_rmse)
    print(f'Model {ens_idx} done: RMSE={best_rmse:.4f}, SR-CI={best_sr_ci:.4f}\n')

print(f'\nEnsemble complete. Individual RMSEs: {[f"{r:.4f}" for r in all_best_rmse]}')
print(f'Mean: {np.mean(all_best_rmse):.4f}')

Training 5-model ensemble...

=== Model 1/5 ===


Ep 01:   0%|          | 0/181 [00:00<?, ?it/s]

Ep 001 | Loss: 0.5532 | RMSE: 0.7685 | SR-CI: 0.7684 (A:0.776 B:0.761 C:0.770)


Ep 005 | Loss: 0.2685 | RMSE: 0.7180 | SR-CI: 0.7905 (A:0.795 B:0.786 C:0.791)


Ep 010 | Loss: 0.2428 | RMSE: 0.7013 | SR-CI: 0.8145 (A:0.818 B:0.808 C:0.818)


...

Ep 080 | Loss: 0.1503 | RMSE: 0.5998 | SR-CI: 0.8417 (A:0.846 B:0.834 C:0.845)


Ep 085 | Loss: 0.1446 | RMSE: 0.5903 | SR-CI: 0.8442 (A:0.845 B:0.839 C:0.848)


Ep 090 | Loss: 0.1409 | RMSE: 0.5813 | SR-CI: 0.8439 (A:0.846 B:0.837 C:0.849)


Ep 095 | Loss: 0.1360 | RMSE: 0.5512 | SR-CI: 0.8550 (A:0.858 B:0.847 C:0.860)


Ep 100 | Loss: 0.1255 | RMSE: 0.5483 | SR-CI: 0.8583 (A:0.859 B:0.853 C:0.863)
Model 5 done: RMSE=0.5483, SR-CI=0.8583


Ensemble complete. Individual RMSEs: ['0.5299', '0.5451', '0.5328', '0.5308', '0.5483']
Mean: 0.5374


## 5. Validate Ensemble

Averaging multiple models reduces variance — their individual errors partially cancel out.

In [15]:
# Load all members and compute ensemble val performance
models = []
for i in range(1, ENSEMBLE_SIZE + 1):
    m = DTA_Model(node_feat=NODE_FEAT, edge_feat=EDGE_FEAT, esm_dim=ESM_DIM,
                  hidden_dim=HIDDEN_DIM, n_layers=N_GIN_LAYERS, dropout=DROPOUT).to(device)
    m.load_state_dict(torch.load(os.path.join(MODEL_DIR, f'model_{i}.pt'), map_location=device))
    m.eval()
    models.append(m)

per_model_preds = [[] for _ in range(ENSEMBLE_SIZE)]
targets = []

with torch.no_grad():
    for batch in val_loader:
        batch = batch.to(device)
        targets.extend(batch.y.cpu().numpy())
        for i, m in enumerate(models):
            per_model_preds[i].extend(m(batch).cpu().numpy())

targets = np.array(targets)
ens_pred = np.mean(per_model_preds, axis=0)
ens_rmse = np.sqrt(np.mean((ens_pred - targets) ** 2))

print(f'Ensemble val RMSE: {ens_rmse:.4f}')
print(f'Individual mean:   {np.mean(all_best_rmse):.4f}')
print(f'Ensemble gain:     {np.mean(all_best_rmse) - ens_rmse:.4f}')
print('\nPhase 3 complete → proceed to Phase 4.')

Ensemble val RMSE: 0.5245
Individual mean:   0.5374
Ensemble gain:     0.0128

Phase 3 complete → proceed to Phase 4.
